In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "albiach2012apes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "22 05 04_Albiach-Serrano, Bugnyar & Call, 2012.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, '22 05 04_Albiach-Serrano, Bugnyar & Call, 2012.csv')
# df.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)


In [3]:
df['study_id']="albiach2012apes"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"subject": "ape",
"sex":"sex_temp"}, inplace=True)

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [4]:
species=[]
for index, row in df.iterrows():
    if not pd.isna(row['species_y']):
        species.append(row['species_y'])
    else:
        species.append(row['species_x'])
df = df.assign(species=species)

In [5]:
# df.columns
df.rename(columns={"ape": "participant",
                   "age":"age_in_years"}, inplace=True)

In [6]:
space_list = ['species','rearing','order']
for x in space_list:
    df[x].replace(' ', '_', inplace=True, regex=True)

In [7]:
albiach2012apes_standardized=df[['study_id','participant', 'age_in_years', 'sex', 'species', 'session', 'trial',
    'family', 'condition', 'order',  'pattern', 'bait', 'choice', 'correct']]
comp_out_path_stand = os.path.join(out_pathway, 'albiach2012apes_standardized.csv')
albiach2012apes_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =albiach2012apes_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
albiach2012apes_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'albiach2012apes_glossary.csv')
albiach2012apes_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
